# DPO training with Qwen 0.5B

###Install required packages

In [1]:
!pip install --no-deps git+https://github.com/lvwerra/trl.git

  Cloning https://github.com/lvwerra/trl.git to /tmp/pip-req-build-nyzoj_5e
  Running command git clone --filter=blob:none --quiet https://github.com/lvwerra/trl.git /tmp/pip-req-build-nyzoj_5e
  Resolved https://github.com/lvwerra/trl.git to commit 0726977a3aaf893e594dc7c64aced8e90770f020
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for trl: filename=trl-0.26.0.dev0-py3-none-any.whl size=502359 sha256=3f653ac3da2713558e6416be42b841d1cc0da1fe4b08ad11ba9bbb264301ae43
  Stored in directory: /tmp/pip-ephem-wheel-cache-ulvg7yb8/wheels/89/88/01/4b0e255f9df2bdc5f1149f8128ceffed64b7a537c23e7fbab4
Successfully built trl


In [2]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 22.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [3]:
# Install required packages
!pip install -q transformers datasets accelerate bitsandbytes trl peft safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 9.5 MB/s eta 0:00:00


In [12]:
!pip install -U bitsandbytes


In [4]:
# Import libraries
from datasets import load_dataset
from trl import DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

In [ ]:
# 0) (Optional) if you need to log into HF to access model
# from huggingface_hub import login
# login()  # then paste your HF token when prompted

###Load the dataset

In [5]:
# Load dataset (subset 1200)
dataset = load_dataset("Intel/orca_dpo_pairs", split="train[:1200]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/196 [00:00<?, ?B/s]

orca_rlhf.jsonl:   0%|          | 0.00/36.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

In [6]:
#Check the dataset
print("Columns:", dataset.column_names)
print("Example row:", dataset[0])

Columns: ['system', 'question', 'chosen', 'rejected']
Example row: {'system': '', 'question': "You will be given a definition of a task first, then some input of the task.\nThis task is about using the specified sentence and converting the sentence to Resource Description Framework (RDF) triplets of the form (subject, predicate object). The RDF triplets generated must be such that the triplets accurately capture the structure and semantics of the input sentence. The input is a sentence and the output is a list of triplets of the form [subject, predicate, object] that capture the relationships present in the sentence. When a sentence has more than 1 RDF triplet possible, the output must contain all of them.\n\nAFC Ajax (amateurs)'s ground is Sportpark De Toekomst where Ajax Youth Academy also play.\nOutput:", 'chosen': '[\n  ["AFC Ajax (amateurs)", "has ground", "Sportpark De Toekomst"],\n  ["Ajax Youth Academy", "plays at", "Sportpark De Toekomst"]\n]', 'rejected': " Sure, I'd be happy

In [7]:
dataset

Dataset({
    features: ['system', 'question', 'chosen', 'rejected'],
    num_rows: 1200
})

In [8]:
# Split the data to test and validation
num_train_rows = 1026

# Split the dataset
train_dataset = dataset.select(range(num_train_rows))
valid_dataset = dataset.select(range(num_train_rows, len(dataset)))

print("Training rows:", len(train_dataset))
print("Validation rows:", len(valid_dataset))


Training rows: 1026
Validation rows: 174


In [ ]:
valid_dataset['rejected'][9]

##Load the model

In [25]:
# 5) Model & tokenizer (Qwen 0.5B instruct)
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # set pad token

###Build prompt for the base model

In [15]:
def build_chat_prompt(example):
    system_msg = example.get("system", "").strip()
    question = example.get("question", "").strip()

    # system message block
    system_block = ""
    if len(system_msg) > 0:
        system_block = f"<|im_start|>system\n{system_msg}<|im_end|>\n"

    # user message block
    user_block = f"<|im_start|>user\n{question}<|im_end|>\n"

    # assistant start block
    assistant_block = "<|im_start|>assistant\n"

    # Final prompt
    prompt = system_block + user_block + assistant_block
    return {"prompt": prompt}


###Build a clean text prompt from (system + question)

In [16]:
def build_prompt(example):
    system_msg = example["system"].strip() if example["system"] else ""
    question = example["question"].strip()

    if len(system_msg) > 0:
        prompt = system_msg + "\n" + question
    else:
        prompt = question

    return {"prompt": prompt}


###Apply the map

In [17]:
train_dataset = train_dataset.map(build_prompt)
valid_dataset = valid_dataset.map(build_prompt)


Map:   0%|          | 0/1026 [00:00<?, ? examples/s]

Map:   0%|          | 0/174 [00:00<?, ? examples/s]

###Build Qwen chat prompt

In [18]:
def build_chat_prompt(example):
    user_prompt = (
        f"<|im_start|>user\n{example['prompt']}<|im_end|>\n"
    )
    assistant_prompt = "<|im_start|>assistant\n"

    return {"chat_prompt": user_prompt + assistant_prompt}


In [19]:
# Set up text generation pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device='cuda')

# Example prompt
prompt = build_chat_prompt(valid_dataset[0])["chat_prompt"]

# Generate output
outputs = generator(prompt, max_length=100, truncation=True, num_return_sequences=1, temperature=0.7)

print(outputs[0]['generated_text'])

Device set to use cuda
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>user
You are a helpful assistant, who always provide explanation. Think like you are answering to a five year old.
Family of 5 Dies After Crashing into Arizona Lake
10/19/2015 AT 11:40 AM EDT

A family of five, including three children, died Sunday after crashing into an Arizona lake, PEOPLE confirms.

Witnesses saw the SUV crash into Tempe Town Lake at around 12:15 a.m., the

reports. "Officers got on scene pretty quick," Tempe police spokesman Michael Pooley said. "When they got on scene, three jumped into the water immediately. One of the fishermen [who saw the car crash into the lake] also jumped into the water."

Together, they were able to pull 1-year-old Zariyah Baxter and 2-year-old Nazyiah Baxter from the car. They then got the children's parents, Glenn Edward Baxter, 27, and Danica Baxter, 25. All four were taken to a nearby hospital immediately, where the parents and Zariyah were pronounced dead.

Nazyiah was initially hospitalized in critical condition, but late

I evaluate the model’s ability to perform the tasks requested by the system for a given question. In the next section I implement prompt tuning to fine-tune the model for these tasks. After prompt tuning, we observe the improvement in the model’s performance in following the system’s instructions.

##Train the model

In [36]:
ft_model_name = model_name.split('/')[1].replace("Instruct", "DPO")

training_args = DPOConfig(
    output_dir=ft_model_name,
    logging_steps=25,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    bf16=True,
    num_train_epochs=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_strategy="epoch",
    eval_strategy="epoch",
    eval_steps=1,
    report_to="none"
)

device = torch.device('cuda')

In [37]:
trainer = DPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    train_dataset= train_dataset,
    eval_dataset= valid_dataset,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
1,0.606100,0.588560,-0.038275,-0.267941,0.942529,0.229666,-221.998184,-304.015747,-2.207726,-2.414683
2,0.534700,0.509796,-0.078996,-0.516473,0.977012,0.437477,-222.405426,-306.501068,-2.210700,-2.410794


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
1,0.606100,0.588560,-0.038275,-0.267941,0.942529,0.229666,-221.998184,-304.015747,-2.207726,-2.414683
2,0.534700,0.509796,-0.078996,-0.516473,0.977012,0.437477,-222.405426,-306.501068,-2.210700,-2.410794
3,0.497500,0.484272,-0.101796,-0.615844,0.971264,0.514048,-222.633423,-307.494781,-2.212514,-2.409720


TrainOutput(global_step=771, training_loss=0.5672593902211863, metrics={'train_runtime': 5838.9946, 'train_samples_per_second': 0.527, 'train_steps_per_second': 0.132, 'total_flos': 0.0, 'train_loss': 0.5672593902211863, 'epoch': 3.0})

# Use the DPO-Tuned Model

In [38]:
# Load the fine-tuned model
ft_model = trainer.model

In [42]:
# Set up text generation pipeline
generator = pipeline("text-generation", model=ft_model, tokenizer=tokenizer, device='cuda')

# Example prompt
prompt = build_chat_prompt(valid_dataset[0])["chat_prompt"]

# Generate output
outputs = generator(prompt, max_length=100, truncation=True, num_return_sequences=1, temperature=0.7)

print(outputs[0]['generated_text'])

Device set to use cuda
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>user
You are a helpful assistant, who always provide explanation. Think like you are answering to a five year old.
Family of 5 Dies After Crashing into Arizona Lake
10/19/2015 AT 11:40 AM EDT

A family of five, including three children, died Sunday after crashing into an Arizona lake, PEOPLE confirms.

Witnesses saw the SUV crash into Tempe Town Lake at around 12:15 a.m., the

reports. "Officers got on scene pretty quick," Tempe police spokesman Michael Pooley said. "When they got on scene, three jumped into the water immediately. One of the fishermen [who saw the car crash into the lake] also jumped into the water."

Together, they were able to pull 1-year-old Zariyah Baxter and 2-year-old Nazyiah Baxter from the car. They then got the children's parents, Glenn Edward Baxter, 27, and Danica Baxter, 25. All four were taken to a nearby hospital immediately, where the parents and Zariyah were pronounced dead.

Nazyiah was initially hospitalized in critical condition, but late